In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlite3

In [3]:
TR_SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/transaktsioonid/source_data/"
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
TR_DB = "v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"
RESULT = "verb_freqs.db"

In [4]:
# meetod käände saamiseks feats väljalt
def get_case(feats) -> str:
    cases = ["nom", 
             "gen", 
             "part", 
             "adit", 
             "ill", 
             "el", 
             "all", 
             "term", 
             "abl",
             "kom",
             "ad",
             "es",
             "abes",
             "tr"]
    
    feats_split = feats.split(",")
    case = ""
    # peaks olema 1 kääne per feats, aga järjekord pole fikseeritud
    for idx, feat in enumerate(feats_split):
        if feat in cases:
            case = feat
            break
    return case

In [5]:
con = sqlite3.connect(f"{TR_SOURCE_DATA_PATH}{TR_DB}")

con.create_function("get_case", 1, get_case)

cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT}" AS result')

cur.execute("""
DROP TABLE IF EXISTS result.verb_freqs
""")

cur.execute(
    """
    CREATE TABLE result.verb_freqs
    AS
    SELECT
        tr_head.verb AS verb,
        tr_head.verb_compound AS verb_compound,
        get_case(tr_row.feats) AS obj_case,
        count(DISTINCT tr_row.lemma) AS freq
    FROM
    (
        SELECT
            head_id,
            feats,
            lemma
        FROM
            transaction_row
        WHERE
            deprel = "obj"
        AND
            get_case(feats) IN ('nom', 'gen', 'part')
    ) AS tr_row
    INNER JOIN
        transaction_head AS tr_head
    ON 
        tr_row.head_id = tr_head.id
    WHERE
        tr_head.verb != 'olema'
    GROUP BY
        verb,
        verb_compound,
        obj_case
    ORDER BY
        verb,
        verb_compound,
        freq DESC
    """
)
con.close()

In [10]:
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{RESULT}")

cur = con.cursor()

cur.execute("""
DROP TABLE IF EXISTS verbs_obj_case_freqs
""")

cur.execute("""CREATE TABLE verbs_obj_case_freqs AS
SELECT
    verb,
    verb_compound,
            
    SUM(freq) AS total,

    SUM(CASE WHEN obj_case = 'nom'
            THEN freq ELSE 0 END) AS freq_nom,

    SUM(CASE WHEN obj_case = 'gen'
            THEN freq ELSE 0 END) AS freq_gen,

    SUM(CASE WHEN obj_case = 'part'
            THEN freq ELSE 0 END) AS freq_par
FROM 
    verb_freqs
GROUP BY
    verb,
    verb_compound
ORDER BY
    total DESC
"""
)

con.close()